In [1]:
import os
import cv2
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

In [2]:
def read_fer_dataset(folder_path):
    data = []

    for root, dirs, files in os.walk(folder_path):
        for filename in files:
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                img_path = os.path.join(root, filename)
                img = cv2.imread(img_path)

                if img is not None:
                    label = os.path.basename(root)
                    data.append((img, label))

    return data

FER_folder_path = r'C:\Users\Alfariza Rasendria\Downloads\Skripsi\Skripsi\FIX SEMHAS\Program\Dataset\Used Dataset\FER2013'

fer_dataset = read_fer_dataset(FER_folder_path)

print(f"FER2013 dataset: {len(fer_dataset)} images")

FER2013 dataset: 35647 images


In [3]:
def is_black_image(image, threshold=5):
    return np.mean(image) < threshold


def find_black_images(data, threshold=5):
    black_images = []

    for img, label in data:
        if is_black_image(img, threshold):
            black_images.append((img, label))

    return black_images


def remove_black_images(data, black_data):
    black_set = set(id(img) for img, _ in black_data)

    return [
        (img, label)
        for img, label in data
        if id(img) not in black_set
    ]


black_images = find_black_images(fer_dataset)

fer_dataset = remove_black_images(
    fer_dataset,
    black_images
)

print(
    f"FER2013 dataset after black image removal: "
    f"{len(fer_dataset)} images"
)

FER2013 dataset after black image removal: 35635 images


In [4]:
def map_kdef_label(label_code):
    mapping = {
        'AF': 'fear',
        'AN': 'angry',
        'DI': 'disgust',
        'HA': 'happy',
        'NE': 'neutral',
        'SA': 'sad',
        'SU': 'surprise'
    }

    return mapping.get(
        label_code.upper(),
        label_code
    )


face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades
    + "haarcascade_frontalface_default.xml"
)


def crop_face(img):
    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5
    )

    if len(faces) > 0:
        x, y, w, h = faces[0]
        return img[y:y+h, x:x+w]

    return img


def read_kdef_frontal_images(folder_path):
    data = []

    for root, dirs, files in os.walk(folder_path):

        for filename in files:

            if filename.lower().endswith(
                ('.jpg', '.jpeg', '.png')
            ):

                name, ext = os.path.splitext(filename)

                if (
                    len(name) > 0
                    and name[-1].upper() == 'S'
                ):

                    emotion_code = name[4:6].upper()
                    label = map_kdef_label(emotion_code)

                    subject_id = name[:4]

                    img_path = os.path.join(
                        root,
                        filename
                    )

                    img = cv2.imread(img_path)

                    if img is not None:

                        face = crop_face(img)

                        data.append(
                            (
                                face,
                                label,
                                subject_id,
                                "KDEF"
                            )
                        )

    return data

In [5]:
KDEF_folder_path = r'C:\Users\Alfariza Rasendria\Downloads\Skripsi\Skripsi\FIX SEMHAS\Program\Dataset\Raw Dataset\KDEF_and_AKDEF\KDEF'

kdef_dataset = read_kdef_frontal_images(
    KDEF_folder_path
)

print(
    f"KDEF dataset (frontal): "
    f"{len(kdef_dataset)} images"
)

KDEF dataset (frontal): 980 images


In [6]:
def read_ckplus_images(folder_path):
    data = []

    label_map = {
        'anger': 'angry',
        'sadness': 'sad'
    }

    for root, dirs, files in os.walk(folder_path):

        for filename in files:

            if filename.lower().endswith(
                ('.jpg', '.jpeg', '.png')
            ):

                img_path = os.path.join(
                    root,
                    filename
                )

                img = cv2.imread(img_path)

                if img is not None:

                    label = os.path.basename(
                        root
                    ).lower()

                    label = label_map.get(
                        label,
                        label
                    )

                    # Extract Subject ID
                    subject_id = filename.split('_')[0]

                    data.append(
                        (
                            img,
                            label,
                            subject_id,
                            "CK+"
                        )
                    )

    return data


CK_folder_path = r'C:\Users\Alfariza Rasendria\Downloads\Skripsi\Skripsi\FIX SEMHAS\Program\Dataset\Raw Dataset\CK+'

ck_dataset = read_ckplus_images(
    CK_folder_path
)


# Remove contempt class
ck_dataset = [
    (
        img,
        label,
        subject_id,
        dataset_name
    )
    for img, label, subject_id, dataset_name
    in ck_dataset
    if label.lower() != "contempt"
]


print(
    f"CK+ dataset (without contempt): "
    f"{len(ck_dataset)} images"
)

CK+ dataset (without contempt): 927 images


In [7]:
combined_metadata = []

# FER2013
for img, label in fer_dataset:
    combined_metadata.append(
        (label, None, "FER2013")
    )


# KDEF
for img, label, subject_id, dataset_name in kdef_dataset:
    combined_metadata.append(
        (label, subject_id, dataset_name)
    )


# CK+
for img, label, subject_id, dataset_name in ck_dataset:
    combined_metadata.append(
        (label, subject_id, dataset_name)
    )


print(
    f"Total combined dataset: "
    f"{len(combined_metadata)} images"
)

Total combined dataset: 37542 images


In [8]:
metadata = pd.DataFrame(
    combined_metadata,
    columns=[
        "label",
        "subject_id",
        "dataset"
    ]
)
metadata["index"] = range(len(metadata))

In [9]:
all_indices = metadata["index"].values
all_labels = metadata["label"].values

In [10]:
train_idx, temp_idx = train_test_split(
    all_indices,    
    test_size=0.3,
    stratify=all_labels,
    random_state=42
)

In [11]:
temp_labels = metadata.loc[
    temp_idx,
    "label"
].values

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=temp_labels,
    random_state=42
)

In [12]:
metadata["split"] = "Train"

metadata.loc[
    metadata["index"].isin(val_idx),
    "split"
] = "Validation"

metadata.loc[
    metadata["index"].isin(test_idx),
    "split"
] = "Test"

split_df = metadata.copy()

In [13]:
print("\n======================================")
print("DATA SPLITTING")
print("======================================")

split_percentage = (
    split_df["split"]
    .value_counts(normalize=True)
    .reindex(["Train", "Validation", "Test"])
    .mul(100)
    .round(2)
)

print(split_percentage)


DATA SPLITTING
split
Train         70.0
Validation    15.0
Test          15.0
Name: proportion, dtype: float64


In [ ]:
results = []

for dataset_name in ["KDEF", "CK+"]:

    # Filter data by dataset
    dataset_split_df = split_df[
        split_df["dataset"] == dataset_name
    ]

    # Get Subject IDs in the training set
    train_subjects = set(
        dataset_split_df[
            dataset_split_df["split"] == "Train"
        ]["subject_id"]
    )

    # Get Subject IDs in the test set
    test_subjects = set(
        dataset_split_df[
            dataset_split_df["split"] == "Test"
        ]["subject_id"]
    )

    # Identify Subject IDs present in both Train and Test
    overlap_ids = train_subjects.intersection(
        test_subjects
    )

    # Count overlapping Subject IDs
    overlapping_subject_count = len(
        overlap_ids
    )

    # Get all images belonging to overlapping Subject IDs
    overlapping_images = dataset_split_df[
        dataset_split_df["subject_id"].isin(
            overlap_ids
        )
    ]

    # Count overlapping images
    overlapping_image_count = len(
        overlapping_images
    )

    # Calculate image overlap percentage
    image_overlap_percentage = (
        overlapping_image_count
        / len(metadata)
        * 100
    )

    results.append({
        "Dataset": dataset_name,
        "Train-Test Overlap Subject ID":
            overlapping_subject_count,
        "Overlapping Images":
            overlapping_image_count,
        "Image Overlap (%)":
            round(image_overlap_percentage, 2)
    })


overlap_train_test_results = pd.DataFrame(results)

# Calculate total
total_row = pd.DataFrame([{
    "Dataset": "Total",
    "Train-Test Overlap Subject ID":
        overlap_train_test_results[
            "Train-Test Overlap Subject ID"
        ].sum(),
    "Overlapping Images":
        overlap_train_test_results[
            "Overlapping Images"
        ].sum(),
    "Image Overlap (%)":
        overlap_train_test_results[
            "Image Overlap (%)"
        ].sum()
}])

overlap_train_test_results = pd.concat(
    [
        overlap_train_test_results,
        total_row
    ],
    ignore_index=True
)

print("\n======================================")
print("TRAIN-TEST SUBJECT ID OVERLAP")
print("======================================")

print(
    overlap_train_test_results.to_string(
        index=False
    )
)


# Identify Subject IDs present in both Train and Test
overlap_ids = train_subjects.intersection(
    test_subjects
)

# Count overlapping Subject IDs
overlapping_subject_count = len(
    overlap_ids
)

# Get all images belonging to overlapping Subject IDs
overlapping_images = dataset_split_df[
    dataset_split_df["subject_id"].isin(
        overlap_ids
    )
]

# Count overlapping images
overlapping_image_count = len(
    overlapping_images
)

# Calculate image overlap percentage relative to the entire combined dataset
image_overlap_percentage = (
    overlapping_image_count
    / len(metadata)
    * 100
)


TRAIN-TEST SUBJECT ID OVERLAP
Dataset  Train-Test Overlap Subject ID  Overlapping Images  Image Overlap (%)
   KDEF                             95                 665               1.77
    CK+                             77                 723               1.93
  Total                            172                1388               3.70
